# Potential flow

AE2130-I, Delft University of Technology.

In [14]:
# page:hide
%pip install -q plotly ipywidgets

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets

In [43]:
# page:hide
from scipy.interpolate import RectBivariateSpline

def rk4_step(f, xn, h):
    """One classical RK4 step for dx/ds = f(x)."""
    k1 = f(xn)
    k2 = f(xn + h/2*k1)
    k3 = f(xn + h/2*k2)
    k4 = f(xn + h*k3)
    return xn + h/6*(k1 + 2*k2 + 2*k3 + k4)

def _trace(f, x0, ds, n_max, inside):
    """March from x0 with RK4 steps of size ds (negative = backwards)."""
    pts = [np.asarray(x0, float)]
    for _ in range(n_max):
        xn  = pts[-1]
        xn1 = rk4_step(f, xn, ds)
        if not inside(xn1) or np.allclose(xn1, xn):
            break
        pts.append(xn1)
    return np.array(pts)

def streamline(x0, uv, xy, ds=0.01, n_fwd=300, n_back=300):
    (u, v), (x, y) = uv, xy
    U = RectBivariateSpline(x[::4], y[::4], u[::4, ::4], kx=3, ky=3)   # cubic interpolation
    V = RectBivariateSpline(x[::4], y[::4], v[::4, ::4], kx=3, ky=3)

    def f(xn):                                     # unit velocity direction
        vel = np.array([U(*xn)[0, 0], V(*xn)[0, 0]])
        sp = np.linalg.norm(vel)
        return vel / sp if sp > 1e-12 else np.zeros(2)

    def inside(xn):
        return x[0] <= xn[0] <= x[-1] and y[0] <= xn[1] <= y[-1]

    fwd  = _trace(f, x0,  ds, n_fwd,  inside)
    back = _trace(f, x0, -ds, n_back, inside)
    return np.vstack([back[::-1], fwd[1:]]).T     # back … x0 … forward


This notebook serves to showcase the potential flow module taught in AE2130-I. It is an interactive notebook, so you can modify the code yourself!

In [16]:
x = np.linspace(-1,1,200)
y = np.linspace(-1,1,200)

X, Y     = np.meshgrid(x,y, indexing='ij')
R, THETA = np.hypot(X,Y), np.arctan2(Y,X)

In [6]:
def uniform(Vinf,angle, x=X,y=Y):
  Vx, Vy = Vinf*np.cos(angle), Vinf*np.sin(angle)

  phi = Vx*x + Vy*y
  psi = Vx*y - Vy*x

  return phi, psi

def source(Lambda, r=R,theta=THETA):
  strength = Lambda/(2*np.pi)

  phi = strength*np.log(r)
  psi = strength*theta

  return phi, psi

def doublet(kappa, r=R,theta=THETA):
  strength = kappa/(2*np.pi)

  phi =  strength*np.cos(theta)/r
  psi = -strength*np.sin(theta)/r

  return phi, psi

def vortex(Gamma, r=R,theta=THETA):
  strength = Gamma/(2*np.pi)

  phi = -strength*theta
  psi =  strength*np.log(r)

  return phi, psi

In [7]:
def ddx(f,x=x): return np.gradient(f, x, axis=0)
def ddy(f,x=y): return np.gradient(f, y, axis=1)

## 1. Uniform flow

TEST TEST

## 2. Uniform + Source Flow

_To be written._

In [71]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (-30, 30, 1)
angle = 0

# page:slider (1, 5, 0.1)
Lambda = 1

In [73]:
phi_1, psi_1 = uniform(Vinf,np.deg2rad(angle))
phi_2, psi_2 = source(Lambda)

phi_t = phi_1 + phi_2
psi_t = psi_1 + psi_2

u = ddx(phi_t)
v = ddy(phi_t)

# page:hide-from
V = np.hypot(u, v)/Vinf
Cp, label, zmin, zmax = 1 - (V/Vinf)**2, "Cp", -3, 1     # pressure coefficient

fig = go.Figure()

## ----------------------- ##
## free-stream streamlines ##
## ----------------------- ##
a  = np.deg2rad(angle)
c  = -0.9*np.array([np.cos(a), np.sin(a)])       # centre of the line, upstream
e  =  0.9*np.array([-np.sin(a), np.cos(a)])      # half-length along the perpendicular
seeds = np.linspace(c - e, c + e, 20)            # (21, 2) points from one end to the other

for x0 in seeds:
    xs, ys = streamline(x0, (u, v), (x, y))
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=1),
                    hoverinfo="skip", showlegend=False)

## ------------------ ##
## source streamlines ##
## ------------------ ##
R, n = 0.05, 8                                   # circle radius, number of seeds
ang = np.linspace(0, 2*np.pi, n, endpoint=False)
ring = np.array([0.0, 0.0]) + R*np.column_stack([np.cos(ang), np.sin(ang)])

for x0 in ring:
    bx, by = streamline(x0, (u, v), (x, y))
    fig.add_scatter(x=bx, y=by, mode="lines", line=dict(color="white", width=1),
                    hoverinfo="skip", showlegend=False)
fig.add_scatter(x=[0], y=[0], mode="markers", marker=dict(size=10, color="red"), name="source")

## ---------------- ##
## stagnation point ##
## ---------------- ##
a = np.deg2rad(angle)
d = Lambda / (2*np.pi*Vinf)
t      = np.array([np.cos(a), np.sin(a)])     # freestream direction
nrm    = np.array([-np.sin(a), np.cos(a)])    # perpendicular to it
x_stag = -d * t                               # stagnation point
eps = 0.01

seeds = [(x_stag - eps*t,   dict(n_fwd=0)),   # incoming arm: trace backwards upstream
         (x_stag + eps*nrm, dict(n_back=0)),  # upper branch: trace forwards
         (x_stag - eps*nrm, dict(n_back=0))]  # lower branch: trace forwards

for x0, opts in seeds:
    bx, by = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=bx, y=by, mode="lines", line=dict(color="white", width=3),
                    showlegend=False)
fig.add_scatter(x=[x_stag[0]], y=[x_stag[1]], mode="markers",
                marker=dict(symbol="star", size=16, color="black",
                            line=dict(color="white", width=1.5)),
                showlegend=False)

# pressure field, centred at Cp = 0
dark_rdbu_r = [[0.0, "#4a86c5"],    # light blue  (Cp = -1, suction)
                [0.5, "#1a1a1a"],   # white       (Cp = 0)
                [1.0, "#d9605f"]]   # light red   (Cp = 1, stagnation)
fig.add_contour(x=x, y=y, z=Cp.T,
                contours=dict(start=-1, end=1, size=0.125, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-1, zmax=1, zmid=0,
                colorbar=dict(title="Cp"), hoverinfo="skip")

fig.update_layout(title="Uniform flow + source", width=700, height=600, showlegend=False,
                  xaxis=dict(range=[-1, 1], constrain="domain"), yaxis=dict(range=[-1, 1], scaleanchor="x"))
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)",   # transparent: shows the page's black
                  plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.show()
# page:hide-to

## 2. Uniform + Source + Sink Flow

_To be written._

## 3. Doublet and flow around a cylinder

_To be written._

## 4. Vortex and lift on a cylinder

_To be written._

## 5. Vortex panel method on a NACA airfoil

_To be written._